# **Capstone Project: [Deteksi Penyakit pada Tanaman Kelapa Sawit]**
**ID Group:** LAI25-RM112

**Anggota Kelompok:**
- A200YBF418_Refanda Surya Saputra - A200YBF418@devacademy.id
- A270YAF435_Risky Fahriza - A270YAF435@devacademy.id
- A528YBF449_Sebastian Luth Hasibuan - A528YBF449@devacademy.id
- A184YBF450_Sefza Auma Alam - A184YBF450@devacademy.id

## **Import Libraries/Packages**

In [ ]:
import os
import random
import shutil
import warnings
import zipfile

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

import tensorflow as tf
from keras import Model, layers
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, Callback, ReduceLROnPlateau
from tensorflow.keras.layers import InputLayer, Conv2D, MaxPooling2D, Dense, Dropout, BatchNormalization, \
    GlobalAveragePooling2D, AveragePooling2D
from tensorflow.keras.models import Sequential, Model, load_model
from tensorflow.keras.preprocessing import image
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.layers import TFSMLayer, Input, Flatten, Rescaling
from tensorflow.keras.callbacks import Callback
from tensorflow.saved_model import save
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.applications.mobilenet_v2 import MobileNetV2, preprocess_input
from tensorflow.keras.applications.resnet50 import ResNet50

from PIL import Image
from skimage import io
from skimage import img_as_ubyte
from skimage.exposure import adjust_gamma
from skimage.transform import rotate, AffineTransform, warp
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.utils import class_weight

from tqdm.notebook import tqdm as tq

import gdown
from google.colab import drive
from IPython.display import FileLink

warnings.simplefilter(action='ignore', category=FutureWarning)

print(tf.__version__)

## **Loading Data**

### **Preparation Data**

#### **Using Kaggle Notebook/Remote Server**

In [ ]:
# Ketika menggunakan Kaggle Notebook
kaggle_input_path = '/kaggle/input/palm-disease-dataset/palm-disease-dataset'

kaggle_output_path = '/kaggle/working/palm-disease-dataset'

if not os.path.exists(kaggle_output_path):
    os.makedirs(kaggle_output_path)

try:
    shutil.copytree(kaggle_input_path, kaggle_output_path, dirs_exist_ok=True)
    print(f"Berhasil menyalin dataset dari '{kaggle_input_path}' ke {kaggle_output_path}")
except Exception as e:
    print(f"Unexpected error: {e}")

### **Checking Dataset**

In [ ]:
# Mengecek label pada dataset
palm_disease_path = "/kaggle/working/palm-disease-dataset"

# Menyimpan label
list_label = os.listdir(palm_disease_path)

# Menampilkan list label
list_label

In [ ]:
# Membuat dictionary untuk menyimpan gambar untuk setiap kelas dalam data
palm_disease_image = {}

# Menentukan path sumber dataset
source_path = "/kaggle/working/palm-disease-dataset/"

for i in os.listdir(source_path):
    palm_disease_image[i] = os.listdir(source_path + i)

# menampilkan secara acak 5 gambar untuk masing-masing kelas
fig, ax = plt.subplots(4, 5, figsize=(15, 20))

for i, label in enumerate(os.listdir(source_path)):
    images = np.random.choice(palm_disease_image[label], 5, replace=False)
    for j, image in enumerate(images):
        img_path = os.path.join(source_path, label, image)
        img = Image.open(img_path)
        ax[i, j].imshow(img)
        ax[i, j].set(xlabel=label, xticks=[], yticks=[])

fig.tight_layout()

In [ ]:
# Membuat fungsi untuk melihat jumlah image tiap-tiap kelas
def get_image_count(path):
    image_count = {}
    for label in os.listdir(path):
        image_count[label] = len(os.listdir(os.path.join(path, label)))
    return image_count


print(get_image_count(source_path))

### **Distribution Plot of Class Dataset**

In [ ]:
def plot_distribution(path):
    # Membuat daftar yang menyimpan data untuk setiap nama file, path file, dan label dalam data
    file_name = []
    labels = []
    full_path = []

    # Mendapatkan nama file gambar, path file, dan label satu per satu dengan looping, dan simpan sebagai DataFrame
    for path, subdirs, files in os.walk(path):
        for name in files:
            file_name.append(name)
            labels.append(path.split('/')[-1])
            full_path.append(os.path.join(path, name))

    distribution_train = pd.DataFrame({
        'path': full_path,
        'file_name': file_name,
        'labels': labels
    })
    for name in files:
        file_name.append(name)
        labels.append(path.split('/')[-1])

    # Plot distribusi gambar setiap kelas
    plt.figure(figsize=(12, 6))
    sns.set_style('darkgrid')
    sns.countplot(data=distribution_train, x='labels', palette='viridis')

    plt.title("Distribusi Jumlah Gambar per Kelas", fontsize=16)
    plt.xlabel("Nama Kelas Penyakit", fontsize=12)
    plt.ylabel("Jumlah Gambar", fontsize=12)
    plt.tight_layout()
    plt.show()


plot_distribution(source_path)

## **Preprocessing Data**

### **Spliting Data**

In [ ]:
# Membuat variabel untuk menampung lokasi folder dari dataset gambar
source_path = '/kaggle/working/palm-disease-dataset/'

file_name = []
labels = []
full_path = []

for path, subdirs, files in os.walk(source_path):
    for name in files:
        file_name.append(name)
        labels.append(path.split('/')[-1])
        full_path.append(os.path.join(path, name))

# Memasukkan variabel yang sudah dikumpulkan pada looping di atas menjadi sebuah DataFrame
image_df = pd.DataFrame({
    'file_name': file_name,
    'labels': labels,
    'full_path': full_path
})

# Melihat jumlah data gambar pada masing-masing label
image_df['labels'].value_counts()

In [ ]:
# variabel yang digunaka pada pemisahan data ini, di mana variabel x = data path dan y = data labels
X = image_df['full_path']
y = image_df['labels']

# Split dataset awal menjadi data train dan temp (test dan validasi)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Menyatukan ke dalam maisng-masing DataFrame
df_train = pd.DataFrame({
    'path': X_train,
    'labels': y_train,
    'set': 'train'
})

df_test = pd.DataFrame({
    'path': X_test,
    'labels': y_test,
    'set': 'test'
})

# Menggabungkan DataFrame df_train, df_test, dan df_validation
df_all = pd.concat([df_train, df_test], ignore_index=True)
print(df_all.groupby(['set', 'labels']).size())

In [ ]:
# Mengecek sampel data
print(df_all.sample(5))

In [ ]:
# Memanggil dataset asli yang berisi keseluruhan data gambar yang sesuai dengan labelnya
datasource_path = '/kaggle/working/palm-disease-dataset/'

# Membuat variabel dataset, tempat menampung data yang telah dilakukan splitting
dataset_path = '/kaggle/working/final-dataset/'

for index, row in tq(df_all.iterrows()):
    file_path = row['path']

    if not os.path.exists(file_path):
        file_path = os.path.join(datasource_path, row['labels'], row['image'].split('.')[0])

    # Membuat direktori tujuan folder
    if not os.path.exists(os.path.join(dataset_path, row['set'], row['labels'])):
        os.makedirs(os.path.join(dataset_path, row['set'], row['labels']))

    # Menentukan tujuan file
    destination_file_name = file_path.split('/')[-1]
    destination_path = os.path.join(dataset_path, row['set'], row['labels'], destination_file_name)

    # Memindahkan file ke direktori tujuan
    if not os.path.exists(destination_path):
        shutil.copy2(file_path, destination_path)

### **Image Data Generator**

In [ ]:
# Mendefinisikan direktori daset training, test, dan validation
TRAIN_DIR = '/kaggle/working/final-dataset/train'
TEST_DIR = '/kaggle/working/final-dataset/test'
VAL_DIR = '/kaggle/working/final-dataset/validation'

# Membuat fungsi untuk menampilkan jumlah data train berdasarkan label
def print_total_train_data(label):
    train_temp = os.path.join(TRAIN_DIR, label)
    train_count = len(os.listdir(train_temp))
    print("Total train data for {} is {}".format(label, train_count))

# Membuat fungsi untuk menampilkan jumlah data test berdasarkan label
def print_total_test_data(label):
    test_temp = os.path.join(TEST_DIR, label)
    test_count = len(os.listdir(test_temp))
    print("Total test data for {} is {}".format(label, test_count))


for label in list_label:
    print_total_train_data(label)
    print_total_test_data(label)
    print("\n")

In [ ]:
# Membuat objek ImageDataGenerator untuk menormalisasikan gambar
train_datagen = ImageDataGenerator(
    rescale=1. / 255,
    rotation_range=15,
    width_shift_range=0.25,
    height_shift_range=0.25,
    shear_range=0.2,
    zoom_range=[0.7, 1.3],
    horizontal_flip=True,
    vertical_flip=False,
    fill_mode='nearest',
    brightness_range=[0.7, 1.3],
    channel_shift_range=50,
    validation_split=0.3,
)

test_datagen = ImageDataGenerator(rescale=1. / 255)

IMG_HEIGHT = 224
IMG_WIDTH = 224
BATCH_SIZE = 32
SEED = 42

train_generator = train_datagen.flow_from_directory(
    TRAIN_DIR,
    seed=SEED,
    target_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=True,
    subset='training'
)

validation_generator = train_datagen.flow_from_directory(
    TRAIN_DIR,
    seed=SEED,
    target_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False,
    subset='validation'
)

test_generator = test_datagen.flow_from_directory(
    TEST_DIR,
    seed=SEED,
    target_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

print(train_generator.class_indices)

## **Building Model**

### **Perparing Callbacks**

In [ ]:
# Membuat custom callback untuk menghentikan model saat akurasi lebih dari 95%
class EpochCallback(Callback):
    def on_epoch_end(self, epoch, logs={}):
        if logs.get('accuracy') > 0.95:
            print("\nAkurasi telah mencapai >95%")
            self.model.stop_training = True

epoch_checkpoint = EpochCallback()

# Membuat callback checkpoint
checkpoint = ModelCheckpoint(
    'best_model.h5',
    monitor='val_loss',
    save_best_only=True,
    mode='min',
    verbose=1
)

# Membuat callback early stopping
early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=15,
    restore_best_weights=True,
    verbose=1
)

# Membuat callback untuk memperkecil learning_rate
reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    patience=15,
    min_lr=0.000001,
    verbose=1
)

### **Calculate Class Weight**

In [ ]:
# Melakukan perhitungan bobot kelas
class_indices_dictionary = train_generator.class_indices
class_count = len(class_indices_dictionary)

class_weight_array = class_weight.compute_class_weight(
    class_weight='balanced',
    classes=np.unique(train_generator.classes),
    y=train_generator.classes
)

class_weight_dictionary = dict(enumerate(class_weight_array))

print("class_indices: ", class_indices_dictionary)
print("class weight: ", class_weight_dictionary)

### **Model Experiment 1**

In [ ]:
# Inisialisasi model sekuensial
model_1 = Sequential()

model_1.add(Input(shape=(224, 224, 3)))

model_1.add(Conv2D(64, (3, 3), activation='relu', padding='same'))
model_1.add(BatchNormalization())
model_1.add(Conv2D(64, (3, 3), activation='relu', padding='same'))
model_1.add(MaxPooling2D(pool_size=(2, 2)))

model_1.add(Conv2D(128, (3, 3), activation='relu', padding='same'))
model_1.add(BatchNormalization())
model_1.add(Conv2D(128, (3, 3), activation='relu', padding='same'))
model_1.add(MaxPooling2D(pool_size=(2, 2)))

model_1.add(Conv2D(256, (3, 3), activation='relu', padding='same'))
model_1.add(BatchNormalization())
model_1.add(Conv2D(256, (3, 3), activation='relu', padding='same'))
model_1.add(MaxPooling2D(pool_size=(2, 2)))

model_1.add(GlobalAveragePooling2D())

model_1.add(Dense(256, activation='relu'))
model_1.add(Dropout(0.5))

model_1.add(Dense(len(list_label), activation='softmax'))

opt = Adam(learning_rate=0.0001)

model_1.compile(
    optimizer=opt,
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# rangkuman arsitektur model
model_1.summary()

In [ ]:
# Fitting / training model
history_model_1 = model_1.fit(
    train_generator,
    class_weight=class_weight_dictionary,
    epochs=50,
    validation_data=validation_generator,
    callbacks=[checkpoint, early_stopping, reduce_lr]
)

### **Model Transfer Learning**

In [ ]:
# Melakukan inisiasi model dasar
base_model_tl = MobileNetV2(weights='imagenet', include_top=False, input_shape=(224, 224, 3))

# Melakukan frezze pada model dasar
base_model_tl.trainable = False

# Membuat model baru
inputs = Input(shape=(224, 224, 3))

scale_layer = Rescaling(scale=1 / 127.5, offset=-1)
x = scale_layer(inputs)

x = base_model_tl(inputs, training=False)
x = GlobalAveragePooling2D()(x)
outputs = Dense(len(list_label), activation='softmax')(x)

model_tl = Model(inputs, outputs)
model_tl.summary(show_trainable=True)

In [ ]:
# Melatih model MobileNetV2 dengan data baru
model_tl.compile(optimizer=Adam(), loss='categorical_crossentropy', metrics=['accuracy'])

In [ ]:
model_tl.fit(
    train_generator,
    class_weight=class_weight_dictionary,
    epochs=50,
    steps_per_epoch=train_generator.samples // BATCH_SIZE,
    validation_steps=validation_generator.samples // BATCH_SIZE,
    validation_data=validation_generator,
    callbacks=[checkpoint, early_stopping, epoch_checkpoint]
)

In [ ]:
base_model_tl.trainable = True
model_tl.summary(show_trainable=True)

model_tl.compile(
    optimizer=Adam(1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

history_model_tl = model_tl.fit(
    train_generator,
    class_weight=class_weight_dictionary,
    steps_per_epoch=train_generator.samples // BATCH_SIZE,
    validation_steps=validation_generator.samples // BATCH_SIZE,
    epochs=50,
    validation_data=validation_generator,
    callbacks=[checkpoint, early_stopping, epoch_checkpoint]
)

## **Evaluating Model**

### **Preparing Evaluating**

In [ ]:
# Membuat fungsi untuk melakukan plot metrik
def plot_model_result(epochs, acc, val_acc, loss, val_loss):
    plt.plot(epochs, acc, 'r')
    plt.plot(epochs, val_acc, 'b')
    plt.title('Training and validation accuracy')
    plt.ylabel('accuracy')
    plt.xlabel('epoch')
    plt.legend(['train', 'validation'], loc='upper left')
    plt.show()

    plt.plot(epochs, loss, 'r')
    plt.plot(epochs, val_loss, 'b')
    plt.title('Training and validation loss')
    plt.ylabel('loss')
    plt.xlabel('epoch')
    plt.legend(['train', 'validation'], loc='upper left')
    plt.show()


# Membuat fungsi untuk melakukan evaluasi model
def print_result_evaluating_model(model):
    # Evaluasi model menggunakan data pengujian
    evaluation_model_1 = model.evaluate(test_generator)

    # Menampilkan hasil evaluasi
    print('Test Loss:', evaluation_model_1[0])
    print('Test Accuracy:', evaluation_model_1[1])


# Membuat fungsi untuk menampilkan confusion matrix
def show_model_report(model):
    test_generator.reset()

    preds_1 = model.predict(test_generator, verbose=0)
    y_pred = preds_1.argmax(axis=1)
    y_true = test_generator.classes

    class_names = list(test_generator.class_indices.keys())

    # Menampilkan confusion matrix
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(12, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
    plt.ylabel('Actual')
    plt.xlabel('Predicted')
    plt.title('Confusion Matrix')
    plt.show()

    # Menampilkan classification report
    print(classification_report(y_true, y_pred, target_names=class_names))

### **Evaluating Model 1**

In [ ]:
# Menyimpan nilai metrik akurasi dan loss function setelah pelatihan
acc_1 = history_model_1.history['accuracy']
val_acc_1 = history_model_1.history['val_accuracy']
loss_1 = history_model_1.history['loss']
val_loss_1 = history_model_1.history['val_loss']

epochs = range(len(acc_1))

In [ ]:
plot_model_result(epochs, acc_1, val_acc_1, loss_1, val_loss_1)
print_result_evaluating_model(model_1)
show_model_report(model_1)

### **Evaluating Model Transfer Learning**

In [ ]:
acc_tl = history_model_tl.history['accuracy']
val_acc_tl = history_model_tl.history['val_accuracy']
loss_tl = history_model_tl.history['loss']
val_loss_tl = history_model_tl.history['val_loss']

epochs = range(len(acc_tl))

plot_model_result(epochs, acc_tl, val_acc_tl, loss_tl, val_loss_tl)
print_result_evaluating_model(model_tl)
show_model_report(model_tl)

## **Inferring Model**

### **Preprocessing Predict Image**

In [ ]:
list_label = ['Fungal Disease', 'Healthy', 'Magnesium Deficiency', 'Scale Insect']

# Membuat fungsi untuk menampilkan gambar yang akan diprediski
def show_image(img_predict_path):
    img = image.load_img(img_predict_path, target_size=(224, 224))
    plt.imshow(img)
    plt.axis("off")
    plt.title("Image Input")
    plt.show()

# Memproses gambar inputan menjadi matriks
def preprocessing_img(img_predict_path):
    img = image.load_img(img_predict_path, target_size=(224, 224))

    img_array = image.img_to_array(img)
    img_array = img_array / 255.0
    img_array = np.expand_dims(img_array, axis=0)

    img_tensor = tf.constant(img_array, dtype=tf.float32)
    return img_tensor

# Membuat fungsi untuk melakukan pemrosesan gambar untuk model MobileNetV2
def preprocessing_img_for_mobile_net(img_predict_path):
    img = image.load_img(img_predict_path, target_size=(224, 224))
    img_array = image.img_to_array(img)
    img_array = np.expand_dims(img_array, axis=0)

    img_tensor = tf.constant(img_array, dtype=tf.float32)
    return img_tensor

# Membuat fungsi untuk melakukan prediksi
def predict_img(inferring_model, img_predict_path, preprocessing_img):
    predict = inferring_model.predict(preprocessing_img(img_predict_path))

    predict_output = None

    if isinstance(predict, tuple):
        predict_output = predict['output_0']
    elif isinstance(predict, list):
        predict_output = predict[0]
    elif isinstance(predict, np.ndarray):
        predict_output = predict

    single_img_probably = predict_output[0]

    class_probably_list = []
    for i, prob in enumerate(single_img_probably):
        class_probably_list.append((list_label[i], prob))

    class_probably_sorted = sorted(class_probably_list, key=lambda x: x[1], reverse=True)
    for class_name, prob in class_probably_sorted:
        print(f"- {class_name}: {prob * 100:.2f}%")

    predicted_class_name = class_probably_sorted[0][0]
    confidence = class_probably_sorted[0][1] * 100

    print(f"\nGambar paling cocok dengan kelas: {predicted_class_name}")
    print(f"Confidence: {confidence:.2f}%")


print(list_label)

### **Download Predict Image from Google Drive**

In [ ]:
# Mengunduh gambar prediksi dari Google Drive
file_predict_id = "1MyxbfVFSxRLmAb4rfHKe78ErtPPdd00Y"
url_predict = f"https://drive.google.com/uc?id={file_predict_id}"
gdown.download(url_predict, quiet=False)

In [ ]:
# Unzip berkas yang telah diunduh dari Google Drive
with zipfile.ZipFile('palm-oil-inference.zip', 'r') as zip_file:
    zip_file.extractall()

print("Berhasil melakukan unzip")

### **Inferring Model**

In [ ]:
# Menampilkan gambar prediksi
img_predict_1 = "palm-test-inference/Healthy.jpg"
show_image(img_predict_1)

# Melakukan prediksi
predict_result_1 = predict_img(model_tl, img_predict_1, preprocessing_img_for_mobile_net)

In [ ]:
# Menampilkan gambar prediksi
img_predict_2 = "palm-test-inference/Magnesium Deficiency.jpg"
show_image(img_predict_2)

# Melakukan prediksi
predict_result_2 = predict_img(model_tl, img_predict_2, preprocessing_img_for_mobile_net)

In [ ]:
# Menampilkan gambar prediksi
img_predict_3 = "palm-test-inference/Scale Insect.jpg"
show_image(img_predict_3)

# Melakukan prediksi
predict_result_3 = predict_img(model_tl, img_predict_3, preprocessing_img_for_mobile_net)

In [ ]:
# Menampilkan gambar prediksi
img_predict_4 = "palm-test-inference/Fungal Disease.jpeg"
show_image(img_predict_4)

# Melakukan prediksi
predict_result_4 = predict_img(model_tl, img_predict_4, preprocessing_img_for_mobile_net)

## **Converting Model**

### **Converting Model to SavedModel Format**

In [ ]:
# Konversi model ke format SavedModel
saved_path = '/kaggle/working/model/saved_model'

save(model_tl, saved_path)

### **Converting SavedModel to ZIP (Kaggle Notebook)**

In [ ]:
model_folder_name = "model/saved_model"
saved_model_path = "/kaggle/working/model/saved_model/"

zip_file_name = "palm-oil-model.zip"
zip_file_path = os.path.join("/kaggle", "working", zip_file_name)

if os.path.exists(saved_model_path):
    print(f"Mengkompres folder '{saved_model_path}' menjadi '{zip_file_name}'")

    working_dir = os.getcwd()
    os.chdir(saved_model_path)

    try:
        !zip -r {zip_file_name} .
        print(f"Berhasil mengkompres model ke: {zip_file_path}")

        # Membuat link download
        display(FileLink(zip_file_name))
    except Exception as e:
        print(f"Unexpected error: {e}")
    finally:
        os.chdir(working_dir)

else:
    print(f"Error: folder model '{model_folder_name}' tidak ditemukan di '{saved_model_path}'")